# Chapter 3 — CLIP: Contrastive Language-Image Pre-training

**Goal**: Train a vision encoder and text encoder jointly so that
matching image-text pairs have similar representations.

## Motivation

A ViT trained only on ImageNet classification learns "what class is this?".
That's not enough for a VLM — we want the vision encoder to understand
rich, open-ended language descriptions.

CLIP's solution: train on 400M (image, caption) pairs from the internet.
The training signal is **contrastive** — not "classify this image" but
"match this image to the right caption among N distractors".

## InfoNCE Loss

For a batch of N pairs {(I₁,T₁), ..., (Iₙ,Tₙ)}:

```
Similarity matrix S[i,j] = f_img(Iᵢ) · f_txt(Tⱼ) / τ

Loss = ½ [ CrossEntropy(S, rows) + CrossEntropy(Sᵀ, cols) ]
         image→text matching       text→image matching
```

Positive pairs are on the diagonal; all others are negatives.
Larger batches = more negatives = harder training signal.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

## 3.1 The Similarity Matrix

In [ ]:
# Visualise what CLIP's similarity matrix looks like
# (before training vs after training)

N = 6   # batch size

# Simulate random (untrained) embeddings
img_emb_random = F.normalize(torch.randn(N, 64), dim=-1)
txt_emb_random = F.normalize(torch.randn(N, 64), dim=-1)
sim_random = img_emb_random @ txt_emb_random.T

# Simulate well-trained embeddings (diagonal >> off-diagonal)
shared = F.normalize(torch.randn(N, 64), dim=-1)
img_emb_trained = F.normalize(shared + 0.1 * torch.randn(N, 64), dim=-1)
txt_emb_trained = F.normalize(shared + 0.1 * torch.randn(N, 64), dim=-1)
sim_trained = img_emb_trained @ txt_emb_trained.T

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, sim, title in zip(axes,
    [sim_random.detach(), sim_trained.detach()],
    ['Before training (random)', 'After training (aligned)']):
    im = ax.imshow(sim, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Text index')
    ax.set_ylabel('Image index')
    plt.colorbar(im, ax=ax)
    # Mark the diagonal (positive pairs)
    for i in range(N):
        ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1,
                     fill=False, edgecolor='black', lw=2))

plt.suptitle('CLIP Similarity Matrix (image_emb @ text_emb.T)', fontsize=14)
plt.tight_layout()
plt.savefig('figures/ch03_similarity_matrix.png', dpi=100)
plt.show()

## 3.2 InfoNCE Loss — Step by Step

In [ ]:
from multimodal_from_scratch.training.losses import contrastive_loss

B = 8
D = 64

# Untrained: random embeddings → high loss
img_emb = F.normalize(torch.randn(B, D), dim=-1)
txt_emb = F.normalize(torch.randn(B, D), dim=-1)
loss_random = contrastive_loss(img_emb, txt_emb)

# Near-perfect alignment: low loss
noise = 0.05 * torch.randn(B, D)
img_emb_aligned = F.normalize(img_emb + noise, dim=-1)
txt_emb_aligned = F.normalize(img_emb - noise, dim=-1)  # same direction
loss_aligned = contrastive_loss(img_emb_aligned, txt_emb_aligned)

import math
print(f"Loss (random embeddings):    {loss_random.item():.3f}")
print(f"Loss (aligned embeddings):   {loss_aligned.item():.3f}")
print(f"Lower bound (perfect align): ~0.0")
print(f"Random baseline:             {math.log(B):.3f}  (= ln(batch_size))")

## 3.3 Full CLIP Model

In [ ]:
from multimodal_from_scratch.multimodal.clip import CLIP, CLIPConfig

# Tiny config for CPU testing
cfg = CLIPConfig(
    img_size=64,
    patch_size=16,
    vision_embed_dim=192,
    vision_depth=3,
    vision_heads=3,
    vocab_size=1000,
    context_len=32,
    text_embed_dim=192,
    text_depth=3,
    text_heads=3,
    embed_dim=128,
)

clip = CLIP(cfg)

# Count parameters
vision_params = sum(p.numel() for p in clip.vision_encoder.parameters())
text_params   = sum(p.numel() for p in clip.text_encoder.parameters())
print(f"Vision encoder: {vision_params:,} params")
print(f"Text encoder:   {text_params:,} params")
print(f"Total:          {vision_params + text_params:,} params")

In [ ]:
# Forward pass
B = 4
images    = torch.randn(B, 3, 64, 64)
input_ids = torch.randint(0, cfg.vocab_size, (B, cfg.context_len))

# EOS positions (each sequence ends at position 20 for this example)
eos_positions = torch.full((B,), 20, dtype=torch.long)

out = clip(images, input_ids, eos_positions)

print(f"Loss: {out['loss'].item():.3f}")
print(f"Expected (random model): {math.log(B):.3f}")
print(f"logits_per_image shape:  {out['logits_per_image'].shape}")

## 3.4 Zero-Shot Classification

After training, CLIP can classify images **without any labelled images**.
Encode class names as text ("a photo of a {class}") and pick the closest.

In [ ]:
# Pseudocode showing zero-shot classification logic
# (requires a real tokenizer and trained model, shown as illustration)

class_names = ["cat", "dog", "car", "airplane"]
templates   = [f"a photo of a {c}" for c in class_names]

print("Zero-shot CLIP classification pipeline:")
print("\n1. Encode all class text prompts:")
for t in templates:
    print(f"   '{t}'")

print("\n2. Encode the query image")
print("\n3. Compute cosine similarity:")
print("   scores = image_embedding @ text_embeddings.T")
print("\n4. Predicted class = argmax(scores)")
print("\nNo labelled images needed! This is the power of CLIP.")

## Summary

CLIP teaches vision and language to speak a **common embedding language**:
- A ViT encodes images → 512-dim embedding
- A Transformer encodes text → 512-dim embedding
- Contrastive (InfoNCE) loss pulls matching pairs together

The CLIP vision encoder is the standard starting point for VLMs like LLaVA.
We **freeze** it and add a projection layer to connect to the language model.

**Next**: Chapter 4 — Vision-Language Model (VLM)